In [0]:
%sql
DROP TABLE IF EXISTS blur_proyecto_final.bronze.sales_raw

# Ingesta datos de Ventas
Este workbook realiza la conexion a kaggle para leer los datos de ventas y realiza la ingesta en el esquema de bronze.

In [0]:
# ============================================================
# BLOQUE 1 - Configuración: parámetros + catálogo en Unity Catalog
# ============================================================

# --- Parámetros del proyecto (widgets) -----------------------------
dbutils.widgets.text("kaggle_json", "/Volumes/blur_proyecto_final/default/files/kaggle.json", "Ruta de su kaggle.json (su carpeta del workspace)")
dbutils.widgets.text("dataset", "datasets/datacertlaboratoria/proyecto-3-segmentacin-de-clientes-en-ecommerce/data", "Dataset de Kaggle ")
dbutils.widgets.text("tabla", "bronze.sales_raw", "Tabla sales war ")

KAGGLE_JSON = dbutils.widgets.get("kaggle_json")
DATASET     = dbutils.widgets.get("dataset")
TABLE_RAW   = dbutils.widgets.get("tabla")

# --- Catálogo y esquemas de Unity Catalog ---------------------------
CATALOG = "blur_proyecto_final"  # cámbienlo si su workspace ya usa otro catálogo

try:
    spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
except Exception as e:
    # Si no tienen permiso para crear catálogos, usamos el catálogo por defecto
    CATALOG = spark.sql("SELECT current_catalog()").first()[0]
    print(f"No se pudo crear el catálogo ({type(e).__name__}). Se usará: {CATALOG}")

for esquema in ("landing", "bronze", "silver", "gold"):
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{esquema}")

# Volumen donde guardamos el CSV descargado de Kaggle (archivos, no tablas)
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.landing.kaggle")

print(f"Catálogo listo: {CATALOG} | Volumen de ingesta: /Volumes/{CATALOG}/landing/kaggle")


In [0]:
# ============================================================
# INSTALACIÓN - Librería oficial de Kaggle (una vez por sesión)
# ============================================================
# kagglehub no viene preinstalada en Databricks: se instala por sesión
# con %pip (buena práctica: %pip, no !pip). La instalación se pierde
# al reiniciar, por eso esta celda se ejecuta siempre antes de la descarga.

%pip install -q kagglehub

In [0]:
# ============================================================
# BLOQUE 1b - Extracción: descargar el dataset con kagglehub
# ============================================================
# La autenticación NUNCA se escribe en el código: usamos el kaggle.json
# que cada uno subió a su carpeta del workspace (ver la guía de arriba).
# La librería kagglehub lo lee desde el directorio KAGGLE_CONFIG_DIR.

import os
import shutil
import kagglehub

# 1) Autenticación: copiamos su kaggle.json (definido en el widget del
#    BLOQUE 1) a donde la librería lo busca
os.makedirs("/tmp/.kaggle", exist_ok=True)
shutil.copy(KAGGLE_JSON, "/tmp/.kaggle/kaggle.json")
os.chmod("/tmp/.kaggle/kaggle.json", 0o600)   # kaggle exige permisos restrictivos
os.environ["KAGGLE_CONFIG_DIR"] = "/tmp/.kaggle"

# 2) Dataset (widget del BLOQUE 1) y volumen de destino en UC
VOLUMEN = f"/Volumes/{CATALOG}/landing/kaggle"   # destino en Unity Catalog

# 3) Descarga (el mismo snippet que muestra la página de Kaggle)
ruta_local = kagglehub.dataset_download(DATASET)
print("Descargado en caché local:", ruta_local)

# 4) Copiamos los archivos del caché al volumen de UC
#    (en compute serverless dbutils.fs.cp no puede leer rutas locales del
#     caché; en cambio /Volumes es accesible como carpeta: copiamos con shutil)
for nombre in os.listdir(ruta_local):
    ruta_archivo = os.path.join(ruta_local, nombre)
    if os.path.isfile(ruta_archivo):
        shutil.copy(ruta_archivo, f"{VOLUMEN}/{nombre}")
        print("Copiado a UC:", nombre)

print("\nArchivos listos en el volumen de ingesta:")
display(dbutils.fs.ls(VOLUMEN))

In [0]:
# ============================================================
# BLOQUE 2 - Bronze: los datos crudos llegan "tal cual"
# ============================================================
# Leemos el CSV que kagglehub dejó en el volumen de landing.
# Bronze NO limpia nada: guarda el dato original para poder reprocesar
# en el futuro. Los CSV llegan con todo como string y con nombres de
# columna que traen puntos y corchetes (así los exporta Kaggle).
#
# El dataset trae además un .json con los mismos datos: usamos el CSV.

import os
from pyspark.sql.types import StructType, StructField, StringType, IntegerType


# Detectamos el CSV que dejó la descarga en el volumen (el nombre del
# archivo depende del dataset de cada equipo, así que no lo escribimos a mano)
ARCHIVO = [f for f in os.listdir(VOLUMEN) if f.lower().endswith(".csv")][0]
print("Archivo a cargar:", ARCHIVO)

schema = StructType([
       StructField("factura", StringType(), True),
      StructField("fecha", StringType(), True),
      StructField("cliente", StringType(), True),
      StructField("pais", StringType(), True),
      StructField("cantidad", StringType(), True),
      StructField("monto", StringType(), True),
])
df_raw = (
    spark.read
#     .option("header", True)          # el CSV trae encabezado
    .option("skipRows", 1)
    .schema(schema)    
    .csv(f"{VOLUMEN}/{ARCHIVO}")     # VOLUMEN quedó definido en el BLOQUE 1b
)

# Guardamos Bronze como tabla Delta gobernada en Unity Catalog
#df_raw2= df_raw.drop([0,1])
headers = ( df_raw.take(1) ) #First two rows to be skipped
df_raw.write.mode("overwrite").saveAsTable(f"{CATALOG}.{TABLE_RAW}")

print(f"Bronze: {df_raw.count()} filas crudas, {len(df_raw.columns)} columnas (todas string, tal cual llegan)")
display(spark.table(f"{CATALOG}.{TABLE_RAW}").limit(5))